# 🎙️ Direct Speech-to-Speech Translation (S2ST) with Fairseq

Notebook này hướng dẫn chi tiết cách chạy pipeline **Direct Speech-to-Speech Translation with Discrete Units (Lee et al., 2021)** sử dụng **Fairseq**.

> 💡 **Tổng quan kiến trúc:**
> * **Âm thanh nguồn (Source Speech)** ➡️ **S2UT Transformer** ➡️ **Discrete Units (HuBERT Clusters)** ➡️ **HiFi-GAN Vocoder** ➡️ **Âm thanh mục tiêu (Target Speech .wav)**.
> * **Môi trường tự động tối ưu hóa:** Khởi tạo môi trường Python 3.10 độc lập tại `/tmp/fairseq_env` (bằng `uv` siêu tốc trong 10 giây), tự động vá lỗi `default_factory` của Hydra và tiền biên dịch Cython C-extensions. Bấm **Run All** là chạy thẳng từ đầu đến cuối không lỗi!
> * **Dữ liệu demo:** Tải 32 clip **LJSpeech thật** (public domain) làm src=tgt identity map — HiFi-GAN chỉ synthesis được speech thật; train ~3000 updates trên T4 (dropout=0, label-smoothing=0 để model ghi nhớ) rồi sinh ra audio có nội dung nghe được. Trước khi train còn có **cổng kiểm tra units ↔ vocoder** (Mục 3.2) để chặn lỗi sai file k-means.

--- 
## 1. Khởi tạo môi trường Python 3.10 & Cài đặt Fairseq

> ⚡ **Cấu hình Kaggle:** Chọn **GPU T4** và bật **Internet: ON**.

In [1]:
# 1.1 Kiểm tra thông tin GPU
!nvidia-smi

Thu Oct 29 10:53:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1.2 Tạo môi trường Python 3.10 độc lập tại /tmp/fairseq_env và cài đặt dependencies
import os
import sys
import subprocess
import threading

ENV_DIR = "/tmp/fairseq_env"
PY_BIN = f"{ENV_DIR}/bin/python"

# Cài uv để tải và thiết lập Python 3.10 độc lập trong 10 giây
!pip install uv --quiet

# Khởi tạo môi trường ảo Python 3.10 chuẩn cho Fairseq + Hydra
if not os.path.exists(PY_BIN):
    print("🚀 Đang khởi tạo môi trường Python 3.10 độc lập tại /tmp/fairseq_env...")
    !uv venv {ENV_DIR} --python 3.10 --quiet

print("⏳ Đang cài đặt PyTorch (CUDA 12.1), setuptools, Cython, pandas và dependencies vào Python 3.10...")
!uv pip install --python {PY_BIN} torch torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet
!uv pip install --python {PY_BIN} setuptools wheel cython "numpy<2.0.0" "omegaconf<2.1" "hydra-core>=1.0.7,<1.1" bitarray regex soundfile sacrebleu sentencepiece editdistance joblib ninja pandas scipy pyyaml --quiet

# Kiểm tra xác nhận phiên bản Python 3.10
!{PY_BIN} --version
print(f"✅ Môi trường Python 3.10 đã sẵn sàng hoạt động tại: {PY_BIN}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 88.4 MB/s eta 0:00:00
🚀 Đang khởi tạo môi trường Python 3.10 độc lập tại /tmp/fairseq_env...
⏳ Đang cài đặt PyTorch (CUDA 12.1), setuptools, Cython, pandas và dependencies vào Python 3.10...
Python 3.10.22
✅ Môi trường Python 3.10 đã sẵn sàng hoạt động tại: /tmp/fairseq_env/bin/python


In [3]:
# 1.3 Clone repository s2st-fairseq và cài đặt Fairseq vào môi trường Python 3.10
REPO_URL = "https://github.com/linhnguyen15492/s2st-fairseq.git"
REPO_NAME = "s2st-fairseq"

# Clone repository về thư mục làm việc
if not os.path.exists("fairseq"):
    if not os.path.exists(REPO_NAME):
        print(f"Đang clone repo {REPO_URL}...")
        !git clone {REPO_URL}
    if os.path.exists(f"{REPO_NAME}/fairseq"):
        %cd {REPO_NAME}/fairseq
else:
    if os.path.basename(os.getcwd()) != "fairseq":
        %cd fairseq

# Vá lỗi tương thích hydra_init() với default_factory trong fairseq/dataclass/initialize.py
init_py_path = "fairseq/dataclass/initialize.py"
if os.path.exists(init_py_path):
    with open(init_py_path, "r", encoding="utf-8") as f:
        code = f.read()
    if "f.default_factory" not in code:
        if "import dataclasses" not in code:
            code = code.replace("import logging", "import dataclasses\nimport logging")
        old_loop = (
            "    for k in FairseqConfig.__dataclass_fields__:\n"
            "        v = FairseqConfig.__dataclass_fields__[k].default"
        )
        new_loop = (
            "    for k, f in FairseqConfig.__dataclass_fields__.items():\n"
            "        v = f.default\n"
            "        if v is dataclasses.MISSING and f.default_factory is not dataclasses.MISSING:\n"
            "            v = f.default_factory()\n"
            "        elif v is dataclasses.MISSING:\n"
            "            continue"
        )
        if old_loop in code:
            code = code.replace(old_loop, new_loop)
            with open(init_py_path, "w", encoding="utf-8") as f:
                f.write(code)
            print("✅ Đã vá lỗi hydra_init() trong fairseq/dataclass/initialize.py!")

# Tiền biên dịch Cython sinh file .cpp tránh lỗi Ninja thiếu data_utils_fast.cpp
print("⏳ Biên dịch Cython cho fairseq C-extensions...")
!{PY_BIN} -m cython --cplus fairseq/data/data_utils_fast.pyx
!{PY_BIN} -m cython --cplus fairseq/data/token_block_utils_fast.pyx

# Cài fairseq editable vào môi trường Python 3.10
print("⏳ Cài đặt Fairseq vào môi trường Python 3.10...")
!uv pip install --python {PY_BIN} --no-build-isolation -e ./ --quiet

# Thiết lập binary đường dẫn
FAIRSEQ_TRAIN = f"{ENV_DIR}/bin/fairseq-train"
FAIRSEQ_GENERATE = f"{ENV_DIR}/bin/fairseq-generate"

# Hàm helper thực thi lệnh an toàn có hiển thị chi tiết log
def exec_cmd(cmd, desc="", show_head=25, show_tail=8, log_file=None):
    print(f"▶️ {desc}...")
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{os.getcwd()}:{env.get('PYTHONPATH', '')}"
    res = subprocess.run(cmd, env=env, text=True, capture_output=True)
    combined = (res.stdout or "") + "\n" + (res.stderr or "")
    # Ghi log ĐẦY ĐỦ (không rút gọn) để đưa vào ZIP kết quả
    if log_file:
        try:
            os.makedirs(os.path.dirname(os.path.abspath(log_file)), exist_ok=True)
            with open(log_file, "w", encoding="utf-8") as lf:
                lf.write(combined)
            print(f"   📝 Đã ghi log đầy đủ: {log_file}")
        except OSError as e:
            print(f"   ⚠️ Không ghi đưủc log {log_file}: {e}")
    if res.returncode != 0:
        print(f"❌ THẤT BẠI: {desc}")
        print("--- STDOUT ---")
        print(res.stdout)
        print("--- STDERR ---")
        print(res.stderr)
        raise RuntimeError(f"Lệnh thất bại với mã lỗi {res.returncode}:\n{res.stderr}")
    lines = [l for l in combined.strip().split("\n") if l.strip()]
    # In đầu (chứa CUDA env / device) + cuối (progress / summary)
    if len(lines) <= show_head + show_tail:
        for l in lines:
            print(f"   {l}")
    else:
        for l in lines[:show_head]:
            print(f"   {l}")
        print(f"   ... ({len(lines) - show_head - show_tail} dòng giữa bị rút gọn) ...")
        for l in lines[-show_tail:]:
            print(f"   {l}")
    # Highlight nếu fairseq report đang train trên GPU
    for l in lines:
        low = l.lower()
        if "cuda" in low and ("tesla" in low or "t4" in low or "gpu" in low or "capabilit" in low):
            print(f"   🟢 {l}")

# Kiểm tra import fairseq và hydra từ Python 3.10
!{PY_BIN} -c "import sys, fairseq, hydra; print('✅ Fairseq (v' + fairseq.__version__ + ') & Hydra OK trên Python ' + sys.version.split()[0] + '!')"

# Verify GPU/ CUDA trong venv Python 3.10 (torch cu121) — nếu False thì train sẽ rơi về CPU
!{PY_BIN} -c "import torch; print('torch', torch.__version__, '| cuda_available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

Đang clone repo https://github.com/linhnguyen15492/s2st-fairseq.git...
Cloning into 's2st-fairseq'...
remote: Enumerating objects: 2021, done.
remote: Counting objects: 100% (2021/2021), done.
remote: Compressing objects: 100% (1537/1537), done.
remote: Total 2021 (delta 441), reused 1998 (delta 420), pack-reused 0 (from 0)
Receiving objects: 100% (2021/2021), 16.38 MiB | 24.86 MiB/s, done.
Resolving deltas: 100% (441/441), done.
/kaggle/working/s2st-fairseq/fairseq
⏳ Biên dịch Cython cho fairseq C-extensions...
⏳ Cài đặt Fairseq vào môi trường Python 3.10...
2026-10-29 10:59:08 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
/kaggle/working/s2st-fairseq/fairseq/fairseq/tasks/multires_hubert_pretraining.py:154: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  dictionaries = [ (Dictionary.load(f"{label_dir}/dict.{label}.txt") if label is not "" else None ) for label in self.cfg.labels]
✅ Fairseq (v0.12.2) & Hydra OK trên Python 3

--- 
## 2. Tải Pre-trained Models

Theo tài liệu `direct_s2st_discrete_units.md`, pipeline cần:
1. **HiFi-GAN Vocoder Checkpoint & Config** (huấn luyện trên LJSpeech với 100 HuBERT units layer 6) để biến discrete units thành sóng âm `.wav`.
2. **HuBERT Base & k-means model** để trích xuất và mã hóa âm thanh đích thành discrete units (100 cụm).

In [4]:
import urllib.request

import hashlib

def _sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def download_file(url: str, output_path: str, fallback_urls=None, expected_sha256: str = None):
    """Tải file từ URL nếu chưa tồn tại. Nếu truyền expected_sha256 thì KIỂM TRA khớp;
    file sai bị xoá ngay (tránh âm thầm dùng bản sai gây lỗi âm thanh)."""
    output_path = os.path.abspath(output_path)
    if os.path.exists(output_path):
        if expected_sha256:
            got = _sha256(output_path)
            if got.lower() == expected_sha256.lower():
                print(f"File đã tồn tại (sha256 OK): {output_path}")
                return
            print(f"⚠️ File tồn tại nhưng sha256 KHÔNG khớp ({got[:16]}...) → tải lại.")
            os.remove(output_path)
        else:
            print(f"File đã tồn tại: {output_path}")
            return
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    urls = [url] + list(fallback_urls or [])
    for u in urls:
        print(f"Đang tải {u} -> {output_path}...")
        try:
            urllib.request.urlretrieve(u, output_path)
        except Exception as e:
            print(f"⚠️ Thất bại ({e}); thử URL tiếp theo...")
            continue
        if expected_sha256:
            got = _sha256(output_path)
            if got.lower() != expected_sha256.lower():
                print(f"❌ sha256 KHÔNG khớp (được {got[:16]}..., mong {expected_sha256[:16]}...) → xoá & thử tiếp.")
                os.remove(output_path)
                continue
        print(f"✅ Hoàn tất tải: {output_path}")
        return
    raise RuntimeError(
        f"KHÔNG thể tải {output_path} từ bất kỳ URL nào.\n"
        "   → Bật Internet trên Kaggle, hoặc tải thủ công rồi đặt đúng đường dẫn.\n"
        "   → Nếu log trên ghi sha256 KHÔNG khớp nhiều lần: hằng số EXPECTED_SHA256 sai, không phải lỗi mạng.\n"
        "   → KHÔNG thay bằng mirror k-means khác (ví dụ yfyeung/...): nhãn cụm bị hoán vị,\n"
        "      vocoder sẽ sinh âm thanh rè/mờ. Dùng đúng file chính thức km100/km.bin."
    )

# Danh sách models cần thiết
MODEL_URLS = {
    "pretrained_models/vocoder/g_00500000": (
        "https://dl.fbaipublicfiles.com/fairseq/speech_to_speech/vocoder/code_hifigan/hubert_base_100_lj/g_00500000",
        [],
    ),
    "pretrained_models/vocoder/config.json": (
        "https://dl.fbaipublicfiles.com/fairseq/speech_to_speech/vocoder/code_hifigan/hubert_base_100_lj/config.json",
        [],
    ),
    "pretrained_models/hubert/hubert_base_ls960.pt": (
        "https://dl.fbaipublicfiles.com/hubert/hubert_base_ls960.pt",
        [],
    ),
    # ⚠️ Dùng ĐÚNG file k-means mà vocoder HiFi-GAN (hubert_base_100_lj) được huấn luyện kèm.
    #    URL cũ (.../gslm/hubert/hubert_base_ls960_L6_km100.bin) trả về 403 và trước đây
    #    fall-back sang bản mirror "yfyeung/...kmeans.100.model" — bản đó CÙNG cụm nhưng
    #    NHÃN bị hoán vị (cos cùng index ~0.20, chỉ ~1% cụm trùng index) nên vocoder nhận
    #    sai mã → âm thanh rè/mờ. KHÔNG dùng mirror. URL chính thức dưới đây trả 200.
    "pretrained_models/hubert/hubert_base_ls960_L6_km100.bin": (
        "https://dl.fbaipublicfiles.com/textless_nlp/gslm/hubert/km100/km.bin",
        [],  # không fallback: nếu sai/không tải được thì PHẢI dừng, không âm thầm dùng bản sai
    ),
}

# sha256 chính thức của file k-means km100/km.bin (308469 bytes) — khóa chống dùng bản sai
EXPECTED_SHA256 = {
    "pretrained_models/hubert/hubert_base_ls960_L6_km100.bin":
        "f14a3104615485381fc489701d6761c9abbdbb0d43607cf55518a1e2891023fe",
}
assert all(len(v) == 64 for v in EXPECTED_SHA256.values()), "EXPECTED_SHA256 phải là 64 ký tự hex"

for target_path, (url, fallbacks) in MODEL_URLS.items():
    download_file(url, target_path, fallback_urls=fallbacks,
                  expected_sha256=EXPECTED_SHA256.get(target_path))

VOCODER_CKPT = os.path.abspath("pretrained_models/vocoder/g_00500000")
VOCODER_CFG = os.path.abspath("pretrained_models/vocoder/config.json")
HUBERT_CKPT = os.path.abspath("pretrained_models/hubert/hubert_base_ls960.pt")
HUBERT_KM = os.path.abspath("pretrained_models/hubert/hubert_base_ls960_L6_km100.bin")
print("\n✅ Tất cả pre-trained models đã sẵn sàng!")


Đang tải https://dl.fbaipublicfiles.com/fairseq/speech_to_speech/vocoder/code_hifigan/hubert_base_100_lj/g_00500000 -> /kaggle/working/s2st-fairseq/fairseq/pretrained_models/vocoder/g_00500000...
✅ Hoàn tất tải: /kaggle/working/s2st-fairseq/fairseq/pretrained_models/vocoder/g_00500000
Đang tải https://dl.fbaipublicfiles.com/fairseq/speech_to_speech/vocoder/code_hifigan/hubert_base_100_lj/config.json -> /kaggle/working/s2st-fairseq/fairseq/pretrained_models/vocoder/config.json...
✅ Hoàn tất tải: /kaggle/working/s2st-fairseq/fairseq/pretrained_models/vocoder/config.json
Đang tải https://dl.fbaipublicfiles.com/hubert/hubert_base_ls960.pt -> /kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960.pt...
✅ Hoàn tất tải: /kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960.pt
Đang tải https://dl.fbaipublicfiles.com/textless_nlp/gslm/hubert/km100/km.bin -> /kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960_L6_km100

--- 
## 3. Cấu trúc thư mục & Chuẩn bị dữ liệu (Data Preparation)

Dữ liệu audio chuẩn gồm 2 thư mục wav 16,000 Hz:
* `$SRC_AUDIO/{split}/{sample_id}.wav` (Âm thanh nguồn)
* `$TGT_AUDIO/{split}/{sample_id}.wav` (Âm thanh mục tiêu)

> 💡 Cell dưới tải 32 clip **LJSpeech thật** (16kHz) — không dùng sine wave vì HuBERT units của sine không match vocoder HiFi-GAN train trên LJSpeech.

In [5]:
import soundfile as sf
import numpy as np
import urllib.request

# Download vài clip LJSpeech thật (public domain) — HiFi-GAN chỉ synthesis được speech thật,
# sine wave toy sẽ cho audio noise/ ngắn vì HuBERT units của sine không match vocoder.
# 40 ứng viên LJSpeech — downloader tự bỏ qua file lỗi. Cần >= 32 clip thành công
# để model có đủ mẫu học identity map tổng quát (ít clip → dễ mode-collapse).
LJ_SAMPLE_IDS = [f"LJ001-{i:04d}" for i in range(1, 41)]
LJ_BASE = "https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs"

def download_ljspeech_samples(raw_dir: str, sample_ids, sr_target: int = 16000):
    """Tải clip LJSpeech (22.05kHz) và resample về 16kHz cho S2UT."""
    os.makedirs(raw_dir, exist_ok=True)
    resample_script = os.path.join(raw_dir, "_resample.py")
    with open(resample_script, "w", encoding="utf-8") as f:
        f.write("""
import sys
import torchaudio
import soundfile as sf
src, dst, sr = sys.argv[1], sys.argv[2], int(sys.argv[3])
wav, orig_sr = torchaudio.load(src)
wav = torchaudio.functional.resample(wav, orig_sr, sr)
sf.write(dst, wav.squeeze(0).numpy(), sr)
""")
    paths = []
    for sid in sample_ids:
        out_path = os.path.join(raw_dir, f"{sid}.wav")
        if os.path.exists(out_path):
            paths.append(out_path)
            continue
        url = f"{LJ_BASE}/{sid}.wav"
        raw_path = os.path.join(raw_dir, f"{sid}_raw.wav")
        print(f"Đang tải {url}...")
        try:
            urllib.request.urlretrieve(url, raw_path)
            subprocess.run(
                [PY_BIN, resample_script, raw_path, out_path, str(sr_target)],
                check=True, capture_output=True,
            )
            os.remove(raw_path)
            paths.append(out_path)
            print(f"  ✅ {sid} → 16kHz")
        except Exception as e:
            print(f"  ⚠️ Bỏ qua {sid}: {e}")
    return paths

def create_speech_dataset(base_dir: str = "speech_dataset", num_samples: int = 8, sr: int = 16000):
    """Tạo dataset demo từ LJSpeech thật (src = tgt để model dễ overfit identity map)."""
    base_dir = os.path.abspath(base_dir)
    raw_dir = os.path.join(base_dir, "raw")
    src_dir = os.path.join(base_dir, "src_audio")
    tgt_dir = os.path.join(base_dir, "tgt_audio")
    
    paths = download_ljspeech_samples(raw_dir, LJ_SAMPLE_IDS, sr_target=sr)  # thử toàn bộ 40 ứng viên
    if len(paths) < num_samples:
        print(f"⚠️ Chỉ tải được {len(paths)}/{num_samples} clips.")
    
    for split in ("train", "dev"):
        os.makedirs(os.path.join(src_dir, split), exist_ok=True)
        os.makedirs(os.path.join(tgt_dir, split), exist_ok=True)
        for idx, p in enumerate(paths):
            wav, file_sr = sf.read(p, dtype="float32")
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
            # Identity mapping: src = tgt → model học copy units, dễ ra audio có nội dung
            sf.write(os.path.join(src_dir, split, f"sample_{idx}.wav"), wav, file_sr)
            sf.write(os.path.join(tgt_dir, split, f"sample_{idx}.wav"), wav, file_sr)
            
    print(f"✅ Đã tạo dataset speech thật tại: {base_dir} ({len(paths)} clips × 2 splits)")
    return src_dir, tgt_dir

DATA_DIR = os.path.abspath("speech_dataset")
SRC_AUDIO, TGT_AUDIO = create_speech_dataset(DATA_DIR, num_samples=32)
DATA_ROOT = os.path.abspath(os.path.join(DATA_DIR, "s2ut_data"))


Đang tải https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs/LJ001-0001.wav...
  ✅ LJ001-0001 → 16kHz
Đang tải https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs/LJ001-0002.wav...
  ✅ LJ001-0002 → 16kHz
Đang tải https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs/LJ001-0003.wav...
  ✅ LJ001-0003 → 16kHz
Đang tải https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs/LJ001-0004.wav...
  ✅ LJ001-0004 → 16kHz
Đang tải https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs/LJ001-0005.wav...
  ✅ LJ001-0005 → 16kHz
Đang tải https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs/LJ001-0006.wav...
  ✅ LJ001-0006 → 16kHz
Đang tải https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs/LJ001-0007.wav...
  ✅ LJ001-0007 → 16kHz
Đang tải https://huggingface.co/datasets/flexthink/ljspeech/resolve/main/wavs/LJ001-0008.wav...
  ✅ LJ001-0008 → 16kHz
Đang tải https://huggingface.co/datasets/flexthi

### 3.1 Trích xuất Discrete Units từ âm thanh đích
Tạo manifest TSV chuẩn và thực thi `quantize_with_kmeans` trên môi trường Python 3.10.

In [6]:
import glob
import re
import shutil
import soundfile as sf


def _natural_key(path):
    """Sort 0,1,2,...,10 thay vì 0,1,10,11,...,2 (sort chuỗi)."""
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", os.path.basename(path))]


def create_wav_manifest(audio_dir: str, manifest_path: str):
    """Tạo manifest TSV mà quantize_with_kmeans yêu cầu (dòng 1: root_dir, dòng 2+: rel_path\tframes).

    Sắp theo natural sort để thứ tự units khớp `sample_0, sample_1, ..., sample_31`.
    `sorted()` thuần cho ra sample_0, sample_1, sample_10, ... → lệch với mọi thứ đọc theo số.
    """
    audio_dir = os.path.abspath(audio_dir)
    manifest_path = os.path.abspath(manifest_path)
    wav_files = sorted(glob.glob(os.path.join(audio_dir, "*.wav")), key=_natural_key)
    with open(manifest_path, "w", encoding="utf-8") as f:
        f.write(audio_dir + "\n")
        for w in wav_files:
            info = sf.info(w)
            rel = os.path.relpath(w, audio_dir)
            f.write(f"{rel}\t{info.frames}\n")
    print(f"✅ Đã tạo manifest: {manifest_path} ({len(wav_files)} files)")
    return manifest_path


def extract_target_units(
    audio_dir: str,
    output_file: str,
    kmeans_model: str,
    acoustic_model: str,
    layer: int = 6,
    feature_type: str = "hubert",
    extension: str = ".wav",
):
    """Trích xuất discrete units từ âm thanh sử dụng HuBERT & k-means model trên Python 3.10."""
    audio_dir = os.path.abspath(audio_dir)
    output_file = os.path.abspath(output_file)
    kmeans_model = os.path.abspath(kmeans_model)
    acoustic_model = os.path.abspath(acoustic_model)

    manifest_path = os.path.join(os.path.dirname(audio_dir), f"{os.path.basename(audio_dir)}_manifest.tsv")
    create_wav_manifest(audio_dir, manifest_path)

    script_path = os.path.abspath("examples/textless_nlp/gslm/speech2unit/clustering/quantize_with_kmeans.py")
    cmd = [
        PY_BIN,
        script_path,
        "--feature_type", feature_type,
        "--kmeans_model_path", kmeans_model,
        "--acoustic_model_path", acoustic_model,
        "--layer", str(layer),
        "--manifest_path", manifest_path,
        "--out_quantized_file_path", output_file,
        "--extension", extension,
    ]
    exec_cmd(cmd, f"Trích xuất discrete units cho {os.path.basename(audio_dir)}")
    print(f"✅ Đã trích xuất units thành công vào: {output_file}")


def read_units_file(unit_file: str):
    """Đọc file units dạng `sample_id|u1 u2 ...` → (ids, danh sách token) ĐÚNG thứ tự dòng."""
    ids, seqs = [], []
    with open(unit_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sample_id, _, unit_str = line.partition("|")
            ids.append(sample_id)
            seqs.append(unit_str.split())
    return ids, seqs


def synthesize_waveform(
    in_code_file: str,
    vocoder_ckpt: str,
    vocoder_cfg: str,
    results_path: str,
    dur_prediction: bool = True,
):
    """HiFi-GAN: file mã (mỗi dòng 1 chuỗi số) → `{i}_pred.wav` (i = index dòng)."""
    script_path = os.path.abspath("examples/speech_to_speech/generate_waveform_from_code.py")
    cmd = [
        PY_BIN,
        script_path,
        "--in-code-file", os.path.abspath(in_code_file),
        "--vocoder", os.path.abspath(vocoder_ckpt),
        "--vocoder-cfg", os.path.abspath(vocoder_cfg),
        "--results-path", os.path.abspath(results_path),
    ]
    if dur_prediction:
        cmd.append("--dur-prediction")
    exec_cmd(cmd, "Tổng hợp âm thanh .wav bằng HiFi-GAN Vocoder", show_head=15, show_tail=5)
    print(f"✅ Đã tổng hợp file âm thanh thành công vào: {results_path}")


def rename_pred_by_id(results_path: str, ids):
    """Đổi `{i}_pred.wav` → `{sample_id}_pred.wav`.

    generate_waveform_from_code.py đặt tên theo index dòng, nên hai thư mục sinh từ hai danh
    sách khác thứ tự (model output theo dev.tsv vs GT units theo manifest) sẽ bị ghép lệch câu.
    Đặt tên theo sample_id thì đối chiếu luôn đúng, không phụ thuộc thứ tự file.
    """
    preds = sorted(glob.glob(os.path.join(results_path, "*_pred.wav")), key=_natural_key)
    numbered = [p for p in preds if os.path.basename(p).split("_", 1)[0].isdigit()]
    if len(numbered) != len(ids):
        raise RuntimeError(
            f"{results_path}: có {len(numbered)} file *_pred.wav nhưng có {len(ids)} sample_id "
            "→ không thể đặt tên theo id. Xoá thư mục kết quả rồi chạy lại bước tổng hợp."
        )
    for path, sample_id in zip(numbered, ids):
        os.replace(path, os.path.join(results_path, f"{sample_id}_pred.wav"))
    print(f"✅ Đã đổi tên {len(ids)} file wav theo sample_id trong {results_path}")


# Trích xuất target units cho từng split
for split in ["train", "dev"]:
    split_audio_dir = os.path.join(TGT_AUDIO, split)
    split_unit_file = os.path.join(TGT_AUDIO, f"{split}.txt")
    extract_target_units(
        audio_dir=split_audio_dir,
        output_file=split_unit_file,
        kmeans_model=HUBERT_KM,
        acoustic_model=HUBERT_CKPT,
        layer=6,
    )


✅ Đã tạo manifest: /kaggle/working/s2st-fairseq/fairseq/speech_dataset/tgt_audio/train_manifest.tsv (40 files)
▶️ Trích xuất discrete units cho train...
   Writing quantized predictions to /kaggle/working/s2st-fairseq/fairseq/speech_dataset/tgt_audio/train.txt
   2026-10-29 11:03:06 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
   2026-10-29 11:03:08 | INFO | __main__ | Namespace(feature_type='hubert', acoustic_model_path='/kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960.pt', layer=6, kmeans_model_path='/kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960_L6_km100.bin', features_path=None, manifest_path='/kaggle/working/s2st-fairseq/fairseq/speech_dataset/tgt_audio/train_manifest.tsv', out_quantized_file_path='/kaggle/working/s2st-fairseq/fairseq/speech_dataset/tgt_audio/train.txt', extension='.wav', channel_id=None, hide_fname=False)
   2026-10-29 11:03:08 | INFO | __main__ | Extr

### 3.2 Cổng kiểm tra Units ↔ Vocoder (chạy TRƯỚC khi train)

Bước này chặn đúng lỗi đã làm audio demo bị rè: **file k-means không cùng "label space" với vocoder**.
`hubert_base_100_lj` chỉ hiểu đúng mã 0–99 của bộ k-means chính thức (`km100/km.bin`). Một file
k-means khác (mirror trên HuggingFace, km500, ...) gom cụm tương tự nhưng **nhãn cụm bị hoán vị**,
nên vocoder nhận sai mã và sinh ra tiếng rè/kim loại dù pipeline không báo lỗi.

Cách kiểm tra (không cần người nghe, ~10 giây): lấy 1 clip đã biết → HuBERT+k-means → units →
HiFi-GAN → **HuBERT+k-means lần nữa** → so khớp chuỗi units. Vocoder hiểu đúng label space thì chuỗi
units sau khi quay vòng gần như trùng khớp; sai label space thì khớp gần như bằng 0. Sai → `raise`
ngay, không mất ~30 phút train rồi mới phát hiện audio rè.


In [7]:
def _frame_accuracy(ref, hyp):
    """Tỉ lệ frame khớp giữa 2 chuỗi unit (frame-synchronous, so từng vị trí)."""
    if not ref or not hyp:
        return 0.0
    n = min(len(ref), len(hyp))
    return sum(1 for a, b in zip(ref[:n], hyp[:n]) if a == b) / max(len(ref), len(hyp))


def verify_unit_vocoder_roundtrip(
    sample_wav: str,
    kmeans_model: str,
    acoustic_model: str,
    vocoder_ckpt: str,
    vocoder_cfg: str,
    layer: int = 6,
    min_frame_acc: float = 0.5,
    work_dir: str = "results/verify_roundtrip",
):
    """Kiểm tra units ↔ vocoder có cùng label space (round-trip: wav → units → wav → units).

    Trả về (frame_acc, len_ratio). Raise RuntimeError nếu frame_acc < min_frame_acc.
    """
    work_dir = os.path.abspath(work_dir)
    audio_dir = os.path.join(work_dir, "audio")
    resynth_dir = os.path.join(work_dir, "resynth")
    for d in (audio_dir, resynth_dir):
        os.makedirs(d, exist_ok=True)

    sample_id = os.path.splitext(os.path.basename(sample_wav))[0]
    shutil.copyfile(os.path.abspath(sample_wav), os.path.join(audio_dir, f"{sample_id}.wav"))

    # 1) units gốc (frame-level, chưa reduce)
    src_units_file = os.path.join(work_dir, "src_units.txt")
    extract_target_units(audio_dir, src_units_file, kmeans_model, acoustic_model, layer=layer)
    _, src_seqs = read_units_file(src_units_file)
    src = src_seqs[0]

    # 2) units → wav. dur_prediction=False để số frame ra ≈ số frame vào (so khớp được từng vị trí)
    code_file = os.path.join(work_dir, "src_codes.txt")
    with open(code_file, "w", encoding="utf-8") as f:
        f.write(" ".join(src) + "\n")
    synthesize_waveform(code_file, vocoder_ckpt, vocoder_cfg, resynth_dir, dur_prediction=False)

    # 3) wav tái tạo → units lần nữa
    recheck_units_file = os.path.join(work_dir, "recheck_units.txt")
    extract_target_units(resynth_dir, recheck_units_file, kmeans_model, acoustic_model, layer=layer)
    _, recheck_seqs = read_units_file(recheck_units_file)
    rechecked = recheck_seqs[0]

    frame_acc = _frame_accuracy(src, rechecked)
    len_ratio = len(rechecked) / len(src) if src else 0.0
    print(f"🔍 Round-trip {sample_id}: {len(src)} → {len(rechecked)} frames | khớp frame = {frame_acc * 100:.1f}%")
    if frame_acc < min_frame_acc:
        raise RuntimeError(
            f"❌ Cổng kiểm tra THẤT BẠI: units từ {os.path.basename(kmeans_model)} chỉ khớp "
            f"{frame_acc * 100:.1f}% khi quay vòng qua vocoder (yêu cầu ≥ {min_frame_acc * 100:.0f}%).\n"
            "   → Vocoder đang hiểu sai label space của bộ k-means này (nhãn cụm bị hoán vị), "
            "âm thanh sinh ra sẽ rè/kim loại.\n"
            "   → Dùng đúng file của Meta: https://dl.fbaipublicfiles.com/textless_nlp/gslm/hubert/km100/km.bin "
            "(sha256 f14a3104...) rồi chạy lại Mục 2 và Mục 3."
        )
    print(f"✅ Cổng kiểm tra ĐẠT: units và vocoder cùng label space (khớp {frame_acc * 100:.1f}%).")
    return frame_acc, len_ratio


# Chạy trên clip đầu tiên đã tải ở Mục 3 (không phụ thuộc kết quả train)
_VERIFY_WAV = os.path.join(os.path.abspath("speech_dataset"), "raw", "LJ001-0001.wav")
if not os.path.exists(_VERIFY_WAV):
    _VERIFY_WAV = os.path.join(TGT_AUDIO, "train", "sample_0.wav")
ROUNDTRIP_FRAME_ACC, ROUNDTRIP_LEN_RATIO = verify_unit_vocoder_roundtrip(
    sample_wav=_VERIFY_WAV,
    kmeans_model=HUBERT_KM,
    acoustic_model=HUBERT_CKPT,
    vocoder_ckpt=VOCODER_CKPT,
    vocoder_cfg=VOCODER_CFG,
    layer=6,
)


✅ Đã tạo manifest: /kaggle/working/s2st-fairseq/fairseq/results/verify_roundtrip/audio_manifest.tsv (1 files)
▶️ Trích xuất discrete units cho audio...
   Writing quantized predictions to /kaggle/working/s2st-fairseq/fairseq/results/verify_roundtrip/src_units.txt
   2026-10-29 11:03:32 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
   2026-10-29 11:03:34 | INFO | __main__ | Namespace(feature_type='hubert', acoustic_model_path='/kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960.pt', layer=6, kmeans_model_path='/kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960_L6_km100.bin', features_path=None, manifest_path='/kaggle/working/s2st-fairseq/fairseq/results/verify_roundtrip/audio_manifest.tsv', out_quantized_file_path='/kaggle/working/s2st-fairseq/fairseq/results/verify_roundtrip/src_units.txt', extension='.wav', channel_id=None, hide_fname=False)
   2026-10-29 11:03:34 | INFO | __main__

### 3.3 Định dạng dữ liệu S2UT & Sinh Config (`prep_s2ut_data.py`)

In [8]:
def prepare_s2ut_dataset(
    source_dir: str,
    target_dir: str,
    output_root: str,
    splits=("train", "dev"),
    reduce_unit: bool = True,
    vocoder_ckpt: str = None,
    vocoder_cfg: str = None,
):
    """Tạo manifest TSV và config.yaml phục vụ huấn luyện mô hình S2UT trên Python 3.10."""
    script_path = os.path.abspath("examples/speech_to_speech/preprocessing/prep_s2ut_data.py")
    cmd = [
        PY_BIN,
        script_path,
        "--source-dir", os.path.abspath(source_dir),
        "--target-dir", os.path.abspath(target_dir),
        "--data-split", *splits,
        "--output-root", os.path.abspath(output_root),
    ]
    if reduce_unit:
        cmd.append("--reduce-unit")
    if vocoder_ckpt:
        cmd.extend(["--vocoder-checkpoint", os.path.abspath(vocoder_ckpt)])
    if vocoder_cfg:
        cmd.extend(["--vocoder-cfg", os.path.abspath(vocoder_cfg)])

    exec_cmd(cmd, "Chuẩn bị manifest và config S2UT")
    print(f"✅ Đã chuẩn bị xong dữ liệu S2UT tại: {output_root}")

# Gọi hàm chuẩn bị dữ liệu S2UT
prepare_s2ut_dataset(
    source_dir=SRC_AUDIO,
    target_dir=TGT_AUDIO,
    output_root=DATA_ROOT,
    splits=("train", "dev"),
    reduce_unit=True,
    vocoder_ckpt=VOCODER_CKPT,
    vocoder_cfg=VOCODER_CFG,
)

▶️ Chuẩn bị manifest và config S2UT...
   Generating manifest...
   Processing train
   Processed 40 samples
   Writing manifest to /kaggle/working/s2st-fairseq/fairseq/speech_dataset/s2ut_data/train.tsv...
   Processing dev
   Processed 40 samples
   Writing manifest to /kaggle/working/s2st-fairseq/fairseq/speech_dataset/s2ut_data/dev.tsv...
   2026-10-29 11:04:01 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
   100%|██████████| 40/40 [00:00<00:00, 5950.21it/s]
✅ Đã chuẩn bị xong dữ liệu S2UT tại: /kaggle/working/s2st-fairseq/fairseq/speech_dataset/s2ut_data


--- 
## 4. Huấn luyện mô hình (Training - Tối ưu cho Kaggle GPU T4)

> 💡 **Đóng gói hàm `train_s2ut_model`:**
> * **Verify GPU** trước khi train (`torch.cuda.is_available()` trong venv) — in rõ tên Tesla T4 nếu OK.
> * Fairseq **tự dùng CUDA** khi available (không cần flag `--cuda`); log “CUDA environments… name = Tesla T4” sẽ hiển thị.
> * Monitor **GPU util + VRAM** mỗi 5s song song trong lúc train.
> * `--fp16` chỉ bật khi có GPU; `--num-workers 4` + `--update-freq 1` giữ GPU luôn bận (fix bottleneck dataloader trước đây gây util 0-12%).
> * Mặc định **3000 updates** trên **32 clips** LJSpeech thật, với **dropout=0 & label-smoothing=0** (bỏ regularization) để model thực sự ghi nhớ identity map → sinh audio có nội dung rõ.
> * Ô phát âm thanh ở Mục 5 sẽ tự **phát hiện mode-collapse** (các file output giống hệt nhau) để kiểm tra model đã học thật chưa.

In [9]:
def train_s2ut_model(
    data_root: str,
    model_dir: str,
    config_yaml: str = "config.yaml",
    target_code_size: int = 100,
    max_tokens: int = 4000,
    update_freq: int = 1,
    max_update: int = 3000,
    lr: float = 0.0005,
    num_workers: int = 4,
):
    """Thực thi huấn luyện S2UT qua fairseq-train của môi trường Python 3.10."""
    model_dir = os.path.abspath(model_dir)
    os.makedirs(model_dir, exist_ok=True)

    # Verify GPU trước khi train — fairseq tự dùng CUDA nếu available, KHÔNG cần flag --cuda
    gpu_check = subprocess.run(
        [PY_BIN, "-c", "import torch; print(torch.cuda.is_available())"],
        text=True, capture_output=True,
    )
    use_gpu = gpu_check.stdout.strip().lower() == "true"
    if use_gpu:
        name = subprocess.run(
            [PY_BIN, "-c", "import torch; print(torch.cuda.get_device_name(0))"],
            text=True, capture_output=True,
        ).stdout.strip()
        print(f"🟢 GPU OK: {name} — fairseq sẽ train trên CUDA + fp16")
    else:
        print(f"🔴 CUDA=False trong venv — train sẽ rơi về CPU (chậm). stderr={gpu_check.stderr.strip()}")

    fairseq_train = os.path.abspath(f"{ENV_DIR}/bin/fairseq-train")
    if not os.path.exists(fairseq_train):
        fairseq_train = "fairseq-train"

    # Dọn checkpoint cũ + kiểm tra disk — Kaggle /kaggle/working ~20GB, checkpoint S2UT ~100-200MB mỗi file
    import shutil
    if os.path.isdir(model_dir):
        for fn in os.listdir(model_dir):
            if fn.endswith((".pt", ".pt.orig")):
                try:
                    os.remove(os.path.join(model_dir, fn))
                except OSError:
                    pass
        print(f"🧹 Đã dọn checkpoint cũ trong {model_dir}")
    du = shutil.disk_usage(model_dir)
    free_gb = du.free / 1024**3
    print(f"💾 Disk free: {free_gb:.1f} GB")
    if free_gb < 2.0:
        raise RuntimeError(f"Disk chỉ còn {free_gb:.1f} GB — cần >= 2GB để train. Hãy xóa output cũ / restart kernel.")

    cmd = [
        fairseq_train, os.path.abspath(data_root),
        "--config-yaml", config_yaml,
        "--task", "speech_to_speech",
        "--target-is-code",
        "--target-code-size", str(target_code_size),
        "--vocoder", "code_hifigan",
        "--criterion", "speech_to_unit",
        "--label-smoothing", "0",
        "--arch", "s2ut_transformer_fisher",
        "--share-decoder-input-output-embed",
        "--dropout", "0",
        "--attention-dropout", "0",
        "--relu-dropout", "0",
        "--train-subset", "train",
        "--valid-subset", "dev",
        "--save-dir", model_dir,
        "--lr", str(lr),
        "--lr-scheduler", "inverse_sqrt",
        "--warmup-init-lr", "1e-7",
        "--warmup-updates", "100",
        "--optimizer", "adam",
        "--adam-betas", "(0.9,0.98)",
        "--clip-norm", "10.0",
        "--max-update", str(max_update),
        "--max-tokens", str(max_tokens),
        "--max-target-positions", "3000",
        "--update-freq", str(update_freq),
        "--required-batch-size-multiple", "1",
        "--save-interval-updates", "100",
        "--keep-interval-updates", "1",
        "--no-epoch-checkpoints",
        "--log-interval", "10",
        "--seed", "1",
        "--num-workers", str(num_workers),
    ]
    if use_gpu:
        cmd.append("--fp16")  # fp16 chỉ ổn định trên GPU; tránh assert trên CPU

    # Monitor GPU song song trong lúc train
    stop_monitor = threading.Event()
    def _monitor_gpu():
        while not stop_monitor.wait(5):
            try:
                r = subprocess.run(
                    ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
                     "--format=csv,noheader,nounits"],
                    text=True, capture_output=True, timeout=3,
                )
                if r.returncode == 0 and r.stdout.strip():
                    util, mem_used, mem_total = [x.strip() for x in r.stdout.strip().split(",")[:3]]
                    print(f"   ⏱ GPU util={util}% | VRAM={mem_used}/{mem_total} MiB", flush=True)
            except Exception:
                pass

    mon = threading.Thread(target=_monitor_gpu, daemon=True)
    mon.start()
    try:
        exec_cmd(
            cmd,
            f"Huấn luyện mô hình S2UT ({max_update} updates)",
            show_head=15,
            show_tail=25,
            log_file=os.path.join(model_dir, "train.log"),  # log đầy đủ → đưủc đóng gói vào ZIP
        )
    finally:
        stop_monitor.set()
        mon.join(timeout=2)
    print(f"✅ Huấn luyện hoàn tất! Checkpoint lưu tại: {model_dir}")

MODEL_DIR = os.path.abspath("checkpoints/s2ut_model")
# Overfit 32 clips thật với dropout=0/label-smoothing=0 để model học copy units → vocoder ra speech có nội dung.
# Paper chính train 400k updates trên Fisher; demo Kaggle T4 ~30-45 phút với 3000 updates để identity map hội tụ.
train_s2ut_model(data_root=DATA_ROOT, model_dir=MODEL_DIR, max_update=3000)


🟢 GPU OK: Tesla T4 — fairseq sẽ train trên CUDA + fp16
🧹 Đã dọn checkpoint cũ trong /kaggle/working/s2st-fairseq/fairseq/checkpoints/s2ut_model
💾 Disk free: 18.2 GB
▶️ Huấn luyện mô hình S2UT (3000 updates)...
   ⏱ GPU util=0% | VRAM=0/15360
0 MiB
   ⏱ GPU util=0% | VRAM=211/15360
7 MiB
   ⏱ GPU util=0% | VRAM=557/15360
0 MiB
   ⏱ GPU util=0% | VRAM=557/15360
0 MiB
   ⏱ GPU util=0% | VRAM=557/15360
0 MiB
   ⏱ GPU util=0% | VRAM=557/15360
0 MiB
   ⏱ GPU util=0% | VRAM=557/15360
0 MiB
   ⏱ GPU util=35% | VRAM=1063/15360
46 MiB
   ⏱ GPU util=0% | VRAM=1105/15360
100 MiB
   ⏱ GPU util=58% | VRAM=1139/15360
35 MiB
   ⏱ GPU util=8% | VRAM=1157/15360
16 MiB
   ⏱ GPU util=0% | VRAM=1157/15360
100 MiB
   ⏱ GPU util=42% | VRAM=1157/15360
28 MiB
   ⏱ GPU util=8% | VRAM=1157/15360
92 MiB
   ⏱ GPU util=0% | VRAM=1157/15360
100 MiB
   ⏱ GPU util=7% | VRAM=1157/15360
15 MiB
   ⏱ GPU util=9% | VRAM=1157/15360
21 MiB
   ⏱ GPU util=6% | VRAM=1157/15360
90 MiB
   ⏱ GPU util=6% | VRAM=1157/15360
35 MiB
  

--- 
## 5. Suy luận (Inference) & Tổng hợp âm thanh đầu ra

Quy trình suy luận gồm 3 bước rõ ràng:
1. `generate_discrete_units`: Dịch âm thanh nguồn ra log giải mã.
2. `extract_units_from_log`: Trích xuất chuỗi target code trực tiếp bằng **Python thuần** (thay thế hoàn toàn lệnh shell phức tạp `grep | sed | sort | cut`).
3. `synthesize_waveform`: Gọi trực tiếp function của HiFi-GAN Vocoder để tạo file `.wav`.

In [10]:
def generate_discrete_units(
    data_root: str,
    checkpoint_path: str,
    results_path: str,
    gen_subset: str = "dev",
    beam: int = 10,
    max_tokens: int = 20000,
):
    """Chạy fairseq-generate để sinh các dự đoán trên Python 3.10."""
    results_path = os.path.abspath(results_path)
    os.makedirs(results_path, exist_ok=True)

    fairseq_gen = os.path.abspath(f"{ENV_DIR}/bin/fairseq-generate")
    if not os.path.exists(fairseq_gen):
        fairseq_gen = "fairseq-generate"

    cmd = [
        fairseq_gen, os.path.abspath(data_root),
        "--config-yaml", "config.yaml",
        "--task", "speech_to_speech",
        "--target-is-code",
        "--target-code-size", "100",
        "--vocoder", "code_hifigan",
        "--path", os.path.abspath(checkpoint_path),
        "--gen-subset", str(gen_subset),
        "--max-tokens", str(max_tokens),
        "--beam", str(beam),
        "--max-len-a", "1",
        "--results-path", results_path,
    ]
    exec_cmd(cmd, f"Suy luận sinh mã discrete units cho {gen_subset}", show_head=25, show_tail=5)
    print(f"✅ Đã sinh log kết quả tại: {results_path}")

def read_tsv_ids(tsv_path: str):
    """Đọc cột `id` của manifest TSV theo ĐÚNG thứ tự dòng (fairseq đánh số D-<index> theo dòng)."""
    with open(tsv_path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        col = header.index("id")
        return [line.rstrip("\n").split("\t")[col] for line in f if line.strip()]


def extract_units_from_log(log_path: str, output_unit_path: str):
    """
    Trích xuất danh sách discrete units từ log fairseq-generate bằng Python thuần.
    Thay thế các lệnh shell không tương thích đa nền tảng (grep/sed/sort/cut).
    """
    hypotheses = []
    empty_count = 0
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            # Dòng dự đoán có tiền tố D-<id>\t<score>\t<tokens>
            if line.startswith("D-"):
                parts = line.strip().split("\t")
                sample_id = int(parts[0].split("-")[1])
                tokens = parts[2].strip() if len(parts) > 2 else ""
                if not tokens:
                    empty_count += 1
                hypotheses.append((sample_id, tokens))

    hypotheses.sort(key=lambda x: x[0])
    with open(output_unit_path, "w", encoding="utf-8") as f:
        for _, tokens in hypotheses:
            f.write(f"{tokens}\n")

    print(f"✅ Đã trích xuất {len(hypotheses)} chuỗi units vào: {output_unit_path}")
    if empty_count:
        # KHÔNG thay bằng unit 0: làm vậy che mất việc model chưa hội tụ và vocoder vẫn "ra tiếng"
        raise RuntimeError(
            f"❌ {empty_count}/{len(hypotheses)} hypothesis RỖNG trong {log_path}.\n"
            "   → Model chưa học được gì (chưa hội tụ) hoặc checkpoint không khớp config data.\n"
            "   → Xem loss trong train.log; tăng --max-update hoặc kiểm tra checkpoint_best/last."
        )
    if hypotheses and hypotheses[0][1]:
        print(f"   Ví dụ units[0]: {hypotheses[0][1][:80]}")


### 5.1 Thực thi pipeline suy luận và nghe âm thanh kết quả

In [11]:
RESULTS_PATH = os.path.abspath("results/s2ut_results")
GEN_SUBSET = "dev"

# fairseq đánh D-<index> theo THỨ TỰ DÒNG của dev.tsv, và generate_waveform_from_code.py cũng đặt
# tên {index}_pred.wav → đọc cột id để đặt lại tên file wav theo sample_id (không lệ thuộc thứ tự).
dev_ids = read_tsv_ids(os.path.join(DATA_ROOT, f"{GEN_SUBSET}.tsv"))
print(f"{GEN_SUBSET}.tsv có {len(dev_ids)} sample: {dev_ids[:5]} ...")

# Tự động tìm checkpoint có sẵn (ưu tiên best, sau đó đến last)
CHECKPOINT_PATH = os.path.join(MODEL_DIR, "checkpoint_best.pt")
if not os.path.exists(CHECKPOINT_PATH):
    CHECKPOINT_PATH = os.path.join(MODEL_DIR, "checkpoint_last.pt")

if os.path.exists(CHECKPOINT_PATH):
    print(f"Sử dụng checkpoint: {CHECKPOINT_PATH}")
    generate_discrete_units(DATA_ROOT, CHECKPOINT_PATH, RESULTS_PATH, gen_subset=GEN_SUBSET)
    log_file = os.path.join(RESULTS_PATH, f"generate-{GEN_SUBSET}.txt")
    unit_file = os.path.join(RESULTS_PATH, f"generate-{GEN_SUBSET}.unit")
    extract_units_from_log(log_file, unit_file)
    synthesize_waveform(unit_file, VOCODER_CKPT, VOCODER_CFG, RESULTS_PATH, dur_prediction=True)
    rename_pred_by_id(RESULTS_PATH, dev_ids)
else:
    print(f"ℹ️ Chưa tìm thấy checkpoint trong {MODEL_DIR}. Vui lòng huấn luyện mô hình ở Mục 4 trước khi chạy suy luận.")

# Tổng hợp thêm audio tham chiếu từ TARGET units gốc (ground-truth) để đối chiếu pipeline vocoder
gt_unit_file = os.path.join(TGT_AUDIO, f"{GEN_SUBSET}.txt")
REF_RESULTS = os.path.abspath("results/ref_gt_units")
if os.path.exists(gt_unit_file):
    print("\n--- Audio tham chiếu từ ground-truth target units (không qua S2UT model) ---")
    # dev.txt có dạng sample_id|u1 u2 ... (frame-level). Giữ sample_id để đặt tên file wav;
    # bỏ tiền tố khi ghi file mã vì vocoder chỉ nhận số thuần.
    # Giữ nguyên frame-level + dur_prediction=False để không upsample thừa.
    gt_ids, gt_seqs = read_units_file(gt_unit_file)
    gt_codes_file = os.path.join(REF_RESULTS, "gt_codes.txt")
    os.makedirs(REF_RESULTS, exist_ok=True)
    with open(gt_codes_file, "w", encoding="utf-8") as f:
        for seq in gt_seqs:
            f.write(" ".join(seq) + "\n")
    synthesize_waveform(gt_codes_file, VOCODER_CKPT, VOCODER_CFG, REF_RESULTS, dur_prediction=False)
    rename_pred_by_id(REF_RESULTS, gt_ids)

    paired = sorted(set(dev_ids) & set(gt_ids), key=_natural_key)
    print(f"\n✅ Đối chiếu được {len(paired)}/{len(dev_ids)} cặp cùng tên <sample_id>_pred.wav:")
    print(f"   model        → {RESULTS_PATH}")
    print(f"   ground-truth → {REF_RESULTS}")
else:
    print(f"ℹ️ Không tìm thấy ground-truth units tại {gt_unit_file}")


dev.tsv có 40 sample: ['sample_21', 'sample_20', 'sample_15', 'sample_19', 'sample_17'] ...
Sử dụng checkpoint: /kaggle/working/s2st-fairseq/fairseq/checkpoints/s2ut_model/checkpoint_best.pt
▶️ Suy luận sinh mã discrete units cho dev...
   2026-10-29 11:29:25 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
   2026-10-29 11:29:29 | INFO | fairseq_cli.generate | {'_name': None, 'common': {'_name': None, 'no_progress_bar': False, 'log_interval': 100, 'log_format': None, 'log_file': None, 'aim_repo': None, 'aim_run_hash': None, 'tensorboard_logdir': None, 'wandb_project': None, 'azureml_logging': False, 'seed': 1, 'cpu': False, 'tpu': False, 'bf16': False, 'memory_efficient_bf16': False, 'fp16': False, 'memory_efficient_fp16': False, 'fp16_no_flatten_grads': False, 'fp16_init_scale': 128, 'fp16_scale_window': None, 'fp16_scale_tolerance': 0.0, 'on_cpu_convert_precision': False, 'min_loss_scale': 0.0001, 'threshold_loss_scale': None, 'amp': Fals

In [12]:
import glob
import numpy as np
import soundfile as sf
import IPython.display as ipd

def play_generated_audio(results_path: str, max_play: int = 4):
    """Phát thử file âm thanh đã dịch. In RMS/peak và TỰ PHÁT HIỆN mode-collapse (file output giống hệt nhau)."""
    generated_audios = sorted(glob.glob(f"{results_path}/*.wav"), key=_natural_key)
    if not generated_audios:
        print("ℹ️ Chưa có file audio được sinh ra.")
        if os.path.isdir(results_path):
            print(f"   Nội dung {results_path}: {os.listdir(results_path)}")
        return

    print(f"🎧 Tìm thấy {len(generated_audios)} file wav trong {results_path}\n")

    # Đọc thống kê TOÀN BỘ file (không chỉ max_play) để phát hiện trùng lặp → mode collapse
    all_stats = []
    for wav_path in generated_audios:
        data, sr = sf.read(wav_path, dtype="float32")
        if data.ndim > 1:
            data = data.mean(axis=1)
        duration = len(data) / sr
        peak = float(np.max(np.abs(data))) if len(data) else 0.0
        rms = float(np.sqrt(np.mean(data ** 2))) if len(data) else 0.0
        all_stats.append((wav_path, sr, duration, peak, rms))

    for i, (wav_path, sr, duration, peak, rms) in enumerate(all_stats[:max_play]):
        data, _ = sf.read(wav_path, dtype="float32")
        if data.ndim > 1:
            data = data.mean(axis=1)
        # Tăng âm lượng nếu quá nhỏ để nghe được (chỉ lúc phát, không ghi đè file)
        play_data = data
        if 0 < peak < 0.05:
            play_data = data * (0.8 / peak)
            boost = True
        else:
            boost = False

        print(f"▶️ [{i}] {os.path.basename(wav_path)} | {duration:.2f}s | sr={sr} | peak={peak:.4f} | RMS={rms:.4f}"
              + (" (boost volume x%.0f)" % (0.8 / peak) if boost else ""))
        if peak < 1e-6:
            print("   ⚠️ Audio gần như IM LẶNG — model chưa học được gì có nghĩa. Thử tăng max_update hoặc kiểm tra units.")
            continue
        ipd.display(ipd.Audio(play_data, rate=sr, autoplay=False))
        print()

    if len(generated_audios) > max_play:
        print(f"   ... và {len(generated_audios) - max_play} file wav khác.")

    # 🔎 Kiểm tra mode-collapse: 2 file có (độ dài, peak, RMS) gần như y hệt → model đang sinh cùng 1 output
    seen, dup_pairs = {}, []
    for wav_path, _, duration, peak, rms in all_stats:
        key = (round(duration, 2), round(peak, 3), round(rms, 3))
        if key in seen:
            dup_pairs.append((os.path.basename(seen[key]), os.path.basename(wav_path)))
        else:
            seen[key] = wav_path
    if dup_pairs:
        print("   ⚠️ PHÁT HIỆN MODE-COLLAPSE: các file output giống hệt nhau → model chưa phân biệt được input.")
        for a, b in dup_pairs:
            print(f"      • {a} == {b}")
        print("      → Tăng max_update / data, hoặc dropout=0 & label-smoothing=0 để model học thật.")

play_generated_audio(RESULTS_PATH)

# Nếu có audio ground-truth tham chiếu thì phát thêm để đối chiếu vocoder
if 'REF_RESULTS' in dir() and os.path.isdir(REF_RESULTS):
    print("\n" + "="*60)
    print("AUDIO THAM CHIẾU (ground-truth units → HiFi-GAN)")
    print("="*60)
    play_generated_audio(REF_RESULTS)

🎧 Tìm thấy 40 file wav trong /kaggle/working/s2st-fairseq/fairseq/results/s2ut_results

▶️ [0] sample_0_pred.wav | 8.80s | sr=16000 | peak=0.9295 | RMS=0.1325



▶️ [1] sample_1_pred.wav | 1.78s | sr=16000 | peak=0.8120 | RMS=0.1097



▶️ [2] sample_2_pred.wav | 9.38s | sr=16000 | peak=0.8855 | RMS=0.1191



▶️ [3] sample_3_pred.wav | 4.66s | sr=16000 | peak=0.8335 | RMS=0.1162



   ... và 36 file wav khác.
   ⚠️ PHÁT HIỆN MODE-COLLAPSE: các file output giống hệt nhau → model chưa phân biệt được input.
      • sample_11_pred.wav == sample_24_pred.wav
      → Tăng max_update / data, hoặc dropout=0 & label-smoothing=0 để model học thật.

AUDIO THAM CHIẾU (ground-truth units → HiFi-GAN)
🎧 Tìm thấy 40 file wav trong /kaggle/working/s2st-fairseq/fairseq/results/ref_gt_units

▶️ [0] sample_0_pred.wav | 9.64s | sr=16000 | peak=0.9448 | RMS=0.1254



▶️ [1] sample_1_pred.wav | 1.88s | sr=16000 | peak=0.7774 | RMS=0.1112



▶️ [2] sample_2_pred.wav | 9.66s | sr=16000 | peak=0.9004 | RMS=0.1230



▶️ [3] sample_3_pred.wav | 5.12s | sr=16000 | peak=0.8229 | RMS=0.1093



   ... và 36 file wav khác.


--- 
### 5.2 Kiểm chứng Chất lượng Âm thanh Gốc qua Speech Resynthesis (Analysis-by-Synthesis)

Đây là phương pháp bài báo ([Lee et al. 2021](https://arxiv.org/abs/2107.05604) - Section 5.1) sử dụng để kiểm chứng chất lượng âm thanh của **Discrete Units + HiFi-GAN Vocoder**:
1. Tải một câu nói tiếng người thật (`16kHz .wav`).
2. Trích xuất **HuBERT layer 6 discrete units** (100 cụm) và gộp các mã trùng lặp liên tiếp (`reduce-unit`).
3. Dùng **HiFi-GAN Vocoder** (`hubert_base_100_lj` với `--dur-prediction`) để tái tạo lại sóng âm.
4. **Kết quả:** Âm thanh tái tạo trong trẻo, rõ từng từ ngữ âm của tiếng người — chứng minh Vocoder và biểu diễn Units hoạt động đúng theo bài báo gốc.

> ⚠️ Đây là **kiểm chứng khâu units → wav** (analysis-by-synthesis), *không* phải bằng chứng mô hình S2UT đã dịch đúng. MOS ~4.0 là con số bài báo báo cáo cho thí nghiệm resynthesis trên dữ liệu Fisher — muốn trích dẫn thì phải tự đo MOS trên chính clip này.

In [13]:
# 1. Tải file âm thanh tiếng người thật (16kHz standard speech sample)
REAL_SPEECH_DIR = os.path.abspath("real_speech_sample")
os.makedirs(REAL_SPEECH_DIR, exist_ok=True)
SAMPLE_WAV = os.path.join(REAL_SPEECH_DIR, "speech_sample.wav")

SAMPLE_URL = "https://raw.githubusercontent.com/espnet/espnet/master/test_utils/ctc_align_test.wav"
download_file(SAMPLE_URL, SAMPLE_WAV, fallback_urls=[
    "https://raw.githubusercontent.com/espnet/espnet/master/test_utils/st_test.wav"
])

# 2. Trích xuất HuBERT discrete units từ giọng nói thật
REAL_RESYNT_DIR = os.path.abspath("results/real_speech_resynthesis")
os.makedirs(REAL_RESYNT_DIR, exist_ok=True)
real_raw_units_file = os.path.join(REAL_RESYNT_DIR, "real_raw_units.txt")

extract_target_units(
    audio_dir=REAL_SPEECH_DIR,
    output_file=real_raw_units_file,
    kmeans_model=HUBERT_KM,
    acoustic_model=HUBERT_CKPT,
    layer=6,
)

# 3. Gộp các mã trùng lặp liên tiếp (reduce-unit) theo đúng chuẩn bài báo
real_reduced_units_file = os.path.join(REAL_RESYNT_DIR, "real_reduced.unit")
with open(real_raw_units_file, "r", encoding="utf-8") as f_in, open(real_reduced_units_file, "w", encoding="utf-8") as f_out:
    for line in f_in:
        line = line.strip()
        if not line:
            continue
        # Tách bỏ tiền tố sample_id| để Vocoder nhận các số nguyên thuần túy
        if "|" in line:
            _, unit_str = line.split("|", 1)
        else:
            unit_str = line
        tokens = unit_str.strip().split()
        if tokens:
            reduced = [tokens[0]] + [t for i, t in enumerate(tokens[1:]) if t != tokens[i]]
            f_out.write(" ".join(reduced) + "\n")

print(f"✅ Đã tạo file reduced discrete units: {real_reduced_units_file}")

# 4. Tái tạo âm thanh bằng HiFi-GAN Vocoder (--dur-prediction)
synthesize_waveform(
    in_code_file=real_reduced_units_file,
    vocoder_ckpt=VOCODER_CKPT,
    vocoder_cfg=VOCODER_CFG,
    results_path=REAL_RESYNT_DIR,
    dur_prediction=True,
)

rename_pred_by_id(REAL_RESYNT_DIR, ["real_speech_resynthesis"])

# 5. Nghe đối chiếu giữa âm thanh gốc và âm thanh tái tạo qua Discrete Units
resynthesized_wavs = sorted(glob.glob(f"{REAL_RESYNT_DIR}/*.wav"))
print("\n--- KẾT QUẢ ĐỐI CHIẾU THÍNH GIÁC ---")
print("🔊 1. Âm thanh tiếng người gốc (Original Human Speech):")
ipd.display(ipd.Audio(SAMPLE_WAV))

if resynthesized_wavs:
    print("\n✨ 2. Âm thanh tái tạo từ HuBERT Discrete Units qua HiFi-GAN Vocoder:")
    ipd.display(ipd.Audio(resynthesized_wavs[0]))
    print("👉 Nhận xét: Âm thanh tái tạo rõ lời → xác nhận khâu units → wav đúng. "
      "Đây KHÔNG phải bằng chứng mô hình S2UT đã dịch đúng (xem ghi chú đầu Mục 5.2).")

Đang tải https://raw.githubusercontent.com/espnet/espnet/master/test_utils/ctc_align_test.wav -> /kaggle/working/s2st-fairseq/fairseq/real_speech_sample/speech_sample.wav...
✅ Hoàn tất tải: /kaggle/working/s2st-fairseq/fairseq/real_speech_sample/speech_sample.wav
✅ Đã tạo manifest: /kaggle/working/s2st-fairseq/fairseq/real_speech_sample_manifest.tsv (1 files)
▶️ Trích xuất discrete units cho real_speech_sample...
   Writing quantized predictions to /kaggle/working/s2st-fairseq/fairseq/results/real_speech_resynthesis/real_raw_units.txt
   2026-10-29 11:30:04 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
   2026-10-29 11:30:06 | INFO | __main__ | Namespace(feature_type='hubert', acoustic_model_path='/kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960.pt', layer=6, kmeans_model_path='/kaggle/working/s2st-fairseq/fairseq/pretrained_models/hubert/hubert_base_ls960_L6_km100.bin', features_path=None, manifest_path='/k


✨ 2. Âm thanh tái tạo từ HuBERT Discrete Units qua HiFi-GAN Vocoder:


👉 Nhận xét: Âm thanh tái tạo rõ lời → xác nhận khâu units → wav đúng. Đây KHÔNG phải bằng chứng mô hình S2UT đã dịch đúng (xem ghi chú đầu Mục 5.2).


---
### 5.3 📦 Đóng gói kết quả train + inference thành ZIP để tải về

Chỉ đóng gói **các thư mục kết quả**, KHÔNG lấy toàn bộ `/kaggle/working`:

| Thư mục trong ZIP | Nội dung | Nguồn |
|---|---|---|
| `model_output/` | audio do mô hình S2UT dịch + `generate-dev.txt/.unit` (tên file = `<sample_id>_pred.wav`) | Mục 5.1 |
| `reference_gt_units/` | audio từ ground-truth units qua HiFi-GAN (đối chiếu vocoder, cùng tên file) | Mục 5.1 |
| `real_speech_resynthesis/` | audio resynthesis tiếng người thật (Analysis-by-Synthesis) | Mục 5.2 |
| `training/` | `train.log` đầy đủ (loss, lr, GPU util, gradient) | Mục 4 |
| `AUDIO_STATS.csv` | duration/peak/RMS từng file → dò mode-collapse nhanh | tự động |
| `README.txt` | mô tả từng thư mục + danh sách file | tự động |

> 🔎 Vì tên file là `<sample_id>_pred.wav` ở cả hai thư mục, có thể ghép cặp model ↔ ground-truth theo tên mà không sợ lệch thứ tự.
> Kết quả cổng kiểm tra round-trip (Mục 3.2) cũng được ghi vào `README.txt` trong ZIP.

> ⚠️ Checkpoint `.pt` (0.5–1 GB) **không** đưủc kèm theo — nếu cần: `package_results_zip(include_checkpoints=True)`.

📥 Trên Kaggle: mở tab **Output** (hoặc nút **Download**) → tải `s2st_results_package.zip`.


In [14]:
# -*- coding: utf-8 -*-
import zipfile
import glob
import os
from datetime import datetime


def _g(name, default=None):
    """Lấy biến tại GLOBAL scope.

    Lỗi cũ: dùng `name in dir()` bên trong hàm → dir() chỉ trảiển scope LOCAL,
    nên biến RESULTS_PATH/REF_RESULTS/... không bao giối đưủc tìm thấy → báo "Chưa tìm thấy thư mục kết quả".
    """
    return globals().get(name, default)


def _files_of(path):
    """Liệt kê file kết quả trong 1 thư mục (wav + units/log), hoặc chính nó nếu là file."""
    if os.path.isfile(path):
        return [path]
    files = sorted(glob.glob(os.path.join(path, "*.wav")))
    extras = sorted(
        f for f in glob.glob(os.path.join(path, "*.*"))
        if f.lower().endswith((".unit", ".txt", ".log", ".json", ".yaml")) and f not in files
    )
    return files + extras


def _audio_stats(wav_path):
    """Trả về (duration, peak, RMS) — để đối chiếu mode-collapse khi xem ZIP."""
    try:  # ưu tiên soundfile + numpy (chạy trên Kaggle)
        import soundfile as sf
        import numpy as np
        data, sr = sf.read(wav_path, dtype="float32")
        if data.ndim > 1:
            data = data.mean(axis=1)
        if not len(data):
            return (0.0, 0.0, 0.0)
        return (len(data) / sr, float(np.max(np.abs(data))), float(np.sqrt(np.mean(data ** 2))))
    except Exception:
        pass
    try:  # Fallback stdlib `wave` — không cần soundfile/numpy (wav PCM 16-bit)
        import wave
        import struct
        with wave.open(wav_path, "rb") as w:
            sr, n, ch, sw = w.getframerate(), w.getnframes(), w.getnchannels(), w.getsampwidth()
            raw = w.readframes(n)
        duration = n / sr if sr else 0.0
        if sw != 2:
            return (duration, None, None)
        samples = struct.unpack("<%dh" % (len(raw) // 2), raw)
        if ch > 1:
            samples = samples[::ch]
        if not samples:
            return (duration, 0.0, 0.0)
        peak = max(abs(s) for s in samples) / 32768.0
        rms = (sum((s / 32768.0) ** 2 for s in samples) / len(samples)) ** 0.5
        return (duration, peak, rms)
    except Exception:
        return None


def package_results_zip(include_checkpoints: bool = False, output_root: str = None):
    """Đóng gói KẾT QUẢ train + inference (KHÔNG lấy toàn bộ working dir) vào 1 ZIP."""
    cwd = os.getcwd()
    if output_root is None:
        # Ưu tiên /kaggle/working (thư mục duy nhất Kaggle cho tải); fallback cwd khi chạy local
        output_root = "/kaggle/working" if os.path.isdir("/kaggle/working") else cwd
    os.makedirs(output_root, exist_ok=True)
    zip_path = os.path.join(output_root, "s2st_results_package.zip")

    # ---- 1) Kết quả inference (Mục 5.1 / 5.2) ----
    packages = {}
    for arc_name, var_name, rel in (
        ("model_output", "RESULTS_PATH", "results/s2ut_results"),
        ("reference_gt_units", "REF_RESULTS", "results/ref_gt_units"),
        ("real_speech_resynthesis", "REAL_RESYNT_DIR", "results/real_speech_resynthesis"),
    ):
        path = _g(var_name) or os.path.join(cwd, *rel.split("/"))
        if os.path.isdir(path):
            packages[arc_name] = path

    # Tự dó thêm mọi results/* khác chứa wav (phòng khi đặt tên khác / thêm bước mới)
    known = {os.path.abspath(p) for p in packages.values()}
    results_root = os.path.join(cwd, "results")
    if os.path.isdir(results_root):
        for sub in sorted(glob.glob(os.path.join(results_root, "*"))):
            if os.path.isdir(sub) and os.path.abspath(sub) not in known \
                    and glob.glob(os.path.join(sub, "*.wav")):
                packages["results_" + os.path.basename(sub)] = sub

    # ---- 2) Kết quả train (Mục 4): train.log (+ checkpoint nếu yêu cầu) ----
    model_dir = _g("MODEL_DIR") or os.path.join(cwd, "checkpoints", "s2ut_model")
    train_log = os.path.join(model_dir, "train.log")
    checkpoints = sorted(glob.glob(os.path.join(model_dir, "*.pt"))) if os.path.isdir(model_dir) else []
    if os.path.isfile(train_log):
        packages["training"] = train_log
    if include_checkpoints:
        for c in checkpoints:
            packages["training_checkpoints/" + os.path.basename(c)] = c

    if not packages:
        print("\u26a0\ufe0f Chưa tìm thấy thư mục kết quả nào để đóng gói.")
        print("   Hãy chạy Mục 4 (train) và Mục 5.1/5.2 (inference) trước. Kiểm tra:")
        for cand in (
            os.path.join(cwd, "results", "s2ut_results"),
            os.path.join(cwd, "results", "ref_gt_units"),
            os.path.join(cwd, "results", "real_speech_resynthesis"),
            model_dir,
        ):
            state = "C\u00d3 T\u1ed4N T\u1ea0I" if os.path.isdir(cand) else "KH\u00d4NG T\u1ed2N T\u1ea0I"
            print(f"   \u2022 {cand}  \u2192 {state}")
        return None

    manifest, stats_rows, n_files = [], [], 0
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for arc_name, src in packages.items():
            for f in _files_of(src):
                if not os.path.isfile(f):
                    continue
                arc = f"{arc_name}/{os.path.basename(f)}"   # dùng '/' để zip đưủc mở trên mọi nên tảng
                zf.write(f, arc)
                n_files += 1
                manifest.append(arc)
                if f.lower().endswith(".wav"):
                    st = _audio_stats(f)
                    if st:
                        stats_rows.append((arc,) + tuple("" if x is None else round(x, 4) for x in st))

        # Thống kê audio (duration/peak/RMS) \u2192 dò nhanh 2 file giống hết nhau (mode-collapse)
        if stats_rows:
            csv_lines = ["file,duration_sec,peak,rms"]
            csv_lines += [",".join(str(x) for x in row) for row in stats_rows]
            zf.writestr("AUDIO_STATS.csv", "\n".join(csv_lines) + "\n")

        # README mô tả từng thư mục + danh sách checkpoint (không kèm weights)
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        if checkpoints:
            ckpt_lines = [
                f"   {os.path.basename(c)} ({os.path.getsize(c) / 1024 ** 2:.0f} MB)" for c in checkpoints
            ]
        else:
            ckpt_lines = ["   (chưa c\u00f3 \u2014 ch\u1ea1y M\u1ee5c 4)"]
        rt_acc = _g("ROUNDTRIP_FRAME_ACC")
        rt_line = (
            f"Round-trip units\u2192wav\u2192units (M\u1ee5c 3.2): {rt_acc * 100:.1f}% frame kh\u1edbp\n\n"
            if rt_acc
            else ""
        )
        readme = (
            "S2ST Fairseq \u2014 K\u1ebft qu\u1ea3 train + inference\n"
            f"Created: {ts}\n\n{rt_line}"
            "model_output/            = audio dịch bởi mô hình S2UT (Mục 5.1)\n"
            "reference_gt_units/      = audio từ ground-truth units qua HiFi-GAN \u2014 tham chiếu vocoder (Mục 5.1)\n"
            "real_speech_resynthesis/ = audio resynthesis tiếng người thật (Mục 5.2)\n"
            "training/                = train.log đầy đủ: loss, lr, GPU util, gradient (Mục 4)\n"
            "AUDIO_STATS.csv          = duration/peak/RMS từng file wav \u2192 dò mode-collapse\n\n"
            "Checkpoint (.pt) KHÔNG kèm vào ZIP (0.5\u20131 GB). \u0110ang làm việc tại:\n"
            + "\n".join(ckpt_lines) + "\n"
            "   (muốn kèm: package_results_zip(include_checkpoints=True))\n\n"
            f"Files ({n_files}):\n" + "\n".join(manifest) + "\n"
        )
        zf.writestr("README.txt", readme)

    size_mb = os.path.getsize(zip_path) / 1024 ** 2
    print(f"\u2705 Đã đóng gói {n_files} file kết qu\u1ea3 \u2192 {zip_path}  ({size_mb:.2f} MB)")
    for arc_name, src in packages.items():
        print(f"   \u2022 {arc_name:24s} \u2192 {src}")
    # Báo các bước chưa chạy (thiếu wav/log)
    for name, rel, hint in (
        ("model_output", "results/s2ut_results", "Mục 5.1"),
        ("reference_gt_units", "results/ref_gt_units", "Mục 5.1"),
        ("real_speech_resynthesis", "results/real_speech_resynthesis", "Mục 5.2"),
    ):
        if name not in packages:
            print(f"   \u26a0\ufe0f Thiếu {name} (\u2192 {rel}) \u2014 chạy {hint} để bổ sung vào ZIP mới.")
    if not include_checkpoints and checkpoints:
        print(f"   \u2139\ufe0f Bỏ qua {len(checkpoints)} checkpoint (.pt) cho ZIP nhẹ \u2014 chạy lại "
              f"package_results_zip(include_checkpoints=True) nếu cần.")
    print("\U0001f4e5 Trên Kaggle: mở tab **Output** (hoặc nút **Download**) rồi tải file `s2st_results_package.zip`.")
    return zip_path


ZIP_PATH = package_results_zip()


✅ Đã đóng gói 87 file kết quả → /kaggle/working/s2st_results_package.zip  (14.60 MB)
   • model_output             → /kaggle/working/s2st-fairseq/fairseq/results/s2ut_results
   • reference_gt_units       → /kaggle/working/s2st-fairseq/fairseq/results/ref_gt_units
   • real_speech_resynthesis  → /kaggle/working/s2st-fairseq/fairseq/results/real_speech_resynthesis
   • training                 → /kaggle/working/s2st-fairseq/fairseq/checkpoints/s2ut_model/train.log
   ℹ️ Bỏ qua 3 checkpoint (.pt) cho ZIP nhẹ — chạy lại package_results_zip(include_checkpoints=True) nếu cần.
📥 Trên Kaggle: mở tab **Output** (hoặc nút **Download**) rồi tải file `s2st_results_package.zip`.
